In [ ]:
import os
import glob
import pandas as pd
from datetime import datetime

In [ ]:
working_folder = "D:/Ctym/p/p174_GemeloDigital_Leon/1_Infobase/Datos Telefonía NOMMON/00275/"
file_name = '00275_odmatrix.csv.gz'
output_folder = 'D:/Ctym/p/p174_GemeloDigital_Leon/5_Trabajo/dev/outputs'
gz_file = os.path.join(working_folder,file_name)
dateparse = lambda x: datetime.strptime(x, '%Y-%m-%d')

# Read main table
dtypes = {
    'date': 'object',
    'period': 'string',
    'origin': 'int64',
    'destination': 'int64',
    'country': 'string',
    'origin_purpose': 'string',
    'destination_purpose': 'string',
    'residence': 'string',
    'distance': 'string',
    'income': 'string',
    'age': 'string',
    'gender': 'string',
    'professional': 'string',
    'trips': 'float64',
    'trips_km': 'float64'
}

raw_data = pd.read_csv(gz_file, sep = '|', dtype= dtypes, encoding='utf-8')
raw_data['date'] = pd.to_datetime(raw_data['date'], format = "%Y-%m-%d")
raw_data.info()
raw_data.head(5)


In [ ]:
# Trip purpose transformation
# When a trip has Home "H" as origin or destination, the trip purpose will always be the other trip purpose.

def trip_purpose(row):
    if pd.isna(row['origin_purpose']) or pd.isna(row['destination_purpose']):
        return ('N/A')
    if row['origin_purpose'] == 'H':
        return row['destination_purpose']
    elif row['destination_purpose'] == 'H':
        return row['origin_purpose']
    else:
        return row['destination_purpose']

raw_data['trip_purpose'] = raw_data.apply(trip_purpose, axis = 1)

u_purpose = raw_data['trip_purpose'].unique()

assert 'H' not in u_purpose, (
    f"There are still some trips labelled as 'H'"
)

In [ ]:
# The fields 'ID' and 'Descrip' from the following JSON file contain the key-value pairs for the origin and destination codes
import json

json_files = glob.glob(os.path.join(working_folder,'*.json'))
with open (json_files[0], 'r', encoding= 'utf-8') as f:
	zoning = json.load(f)

	# Extract the 'features' list.
	zoning_dict = {
		  feature["properties"]["ID"]: feature["properties"]["Descrip"]
		  for feature in zoning.get("features", [])
	}

	macrozonas = {
		feature["properties"]["ID"]: feature["properties"]["MACROZONA"]
		  for feature in zoning.get("features", [])
	}

	nom_macro = {
		feature["properties"]["MACROZONA"]: feature["properties"]["NOM_MACRO"]
		  for feature in zoning.get("features", [])
	}


# Convert the dictionary to a pandas Series
zoning_series = pd.Series(zoning_dict, name='area_name')
macro_series = pd.Series(macrozonas, name='macrozona')
nom_macro_series = pd.Series(nom_macro, name='nom_macro')

#display(zoning_series.unique())

data = raw_data.copy()

# Map the origin and destination group names
# These groups are used to exclude relations outside the study area
data['origin_gr'] = data['origin'].map(zoning_series)
data['destination_gr'] = data['destination'].map(zoning_series)

# Map the origin and destination macro codes
# The macro codes are used to match the transport model in Visum
data['macro_origin'] = data['origin'].map(macro_series)
data['macro_destination'] = data['destination'].map(macro_series)

display(data.head(2))


In [ ]:
# Add the ordered zone names for the study area
orden_zonas = pd.read_csv('D:/Ctym/p/p174_GemeloDigital_Leon/5_Trabajo/dev/orden_zonas.csv', encoding='utf-8', index_col= 'nombre')

origin = data['origin_gr'].map(orden_zonas['orden'])
dest = data['destination_gr'].map(orden_zonas['orden'])

data['origen_ordenado'] = origin
data['destino_ordenado'] = dest

In [ ]:
# Exclude relations between zones outside the study area (zones 5 in advance)
# Instead of eliminating records from the matrix, we set them equal to 0

data['OD_concat'] = data['origen_ordenado'].astype(str) + '_' + data['destino_ordenado'].astype(str)
exterior_zones = [
    "5-5","5-6","5-7","5-8","5-9",
    "6-5","6-6","6-7","6-8","6-9",
    "7-5","7-6","7-7","7-8","7-9",
    "8-5","8-6","8-7","8-8","8-9",
    "9-5","9-6","9-7","9-8","9-9"
]

# Use .loc to conditionally set matrix elements to 0
data.loc[data['OD_concat'].isin(exterior_zones), 'trips'] = 0

In [ ]:
display(data.head())

In [84]:
for feature in data.columns:
    if feature in ['macro_origin', 'macro_destination', 'date', 'professional', 'trip_purpose', 'income', 'age']:
        print(feature)
        print(data[feature].unique())

date
<DatetimeArray>
['2023-01-08 00:00:00', '2023-01-21 00:00:00', '2023-02-24 00:00:00',
 '2023-05-24 00:00:00', '2023-07-19 00:00:00', '2023-08-25 00:00:00',
 '2023-08-26 00:00:00', '2023-11-17 00:00:00', '2023-12-04 00:00:00']
Length: 9, dtype: datetime64[ns]
income
<StringArray>
['I4', 'I3', 'I2', 'I1', <NA>, 'I0']
Length: 6, dtype: string
age
<StringArray>
['65-100', '25-45', '0-25', '45-65', <NA>]
Length: 5, dtype: string
professional
<StringArray>
['No', 'Urban', <NA>, 'Interurban']
Length: 4, dtype: string
trip_purpose
['O' 'NF' 'W' 'N/A']
macro_origin
[  1  10  80  81  82  11  83  84  12  85  86  13  87  14  88  15  89  90
  91  92  93  94  95  96  97  98  16  99 100 101 102 103 104 105 106 107
 108  17 109 110 111 112 113 114 115 116 117 118  18 119 120 121 122 123
 124 125 126  19   2  20  21  22  23  24  25  26  27  28  29   3  30  31
  32  33  34  35  36  37  38  39   4  40  41  42  43  44  45  46  47  48
  49   5  50  51  52  53  54  55  56  57  58   6  60  61  62  63  6

In [89]:
# Dictionary to store matrices for each day and segment
# Segments split professionals vs non professionals.
# There is further segmentation for non-professionals: trip purpose, income, age
matrices = {}
day_matrices = {}
all_origins = set(data['macro_origin'])

for date in data['date'].unique():
	total_trips_seg = 0
	
	# slice by date
	demand_by_date = data[data['date'] == date]
	total_trips_day = demand_by_date['trips'].sum()
	
	# Filter for N/As in any column
	f_demand_by_date = demand_by_date.dropna()
	total_trips_f = f_demand_by_date['trips'].sum()
	print(f'Total trips: {round(total_trips_day,0)}, total trips in rows without any N/A: {round(total_trips_f,0)}')
	print(f'Trips lost: {round(total_trips_day - total_trips_f,0)}')
	
	# Grouping
	grouped = f_demand_by_date.groupby(['macro_origin', 'macro_destination']).agg(
	trips=('trips', 'sum')
	)
	total_trips_macro_g = grouped['trips'].sum()
	assert total_trips_f.round(4) == total_trips_macro_g.round(4), (
	f"The groupby function results in a different total for date {date}: "
	f"before={round(total_trips_f,0)}, after={round(total_trips_macro_g,0)}"
	)
	
	# Pivot before slicing to check total after all slicings are complete
	matrix = grouped.reset_index().pivot(index = 'macro_origin', columns = 'macro_destination', values = 'trips').fillna(0)
	matrix = matrix.reindex(index=all_origins, columns=all_origins, fill_value=0)
	day_total = matrix.values.sum()
	assert round(total_trips_macro_g,4) == round(day_total,4), (
		f"The pivot function results in a different total:"
		f"Total trips before pivot: {round(total_trips_macro_g,0)}, after: {round(day_total,0)}" 
	)
	key = f"{date.date()}"
	day_matrices[key] = matrix
	
	print(f"Completed overall matrix for date:{date}. Total trips: {round(day_total,0)}")
	
	# slice by professional/non-professional
	for professional in f_demand_by_date['professional'].unique():
		sliced_df = f_demand_by_date[f_demand_by_date['professional'] == professional]
		if pd.notna(professional):
			# slice non-professionals further
			if professional == 'No':
				for purpose in sliced_df['trip_purpose'].unique():
					if pd.notna(purpose):
						sliced_df_purpose = sliced_df[sliced_df['trip_purpose'] == purpose]
						for income in sliced_df_purpose['income'].unique():
							if pd.notna(income):
								sliced_df_purpose_income = sliced_df_purpose[sliced_df_purpose['income'] == income]
								for age in sliced_df_purpose_income['age'].unique():
									if pd.notna(age):
										sliced_df_purpose_income_age = sliced_df_purpose_income[sliced_df_purpose_income['age'] == age]
										total_trips_sliced = sliced_df_purpose_income_age['trips'].sum()
										# reshape
										grouped = sliced_df_purpose_income_age.groupby(['macro_origin', 'macro_destination']).agg(
										trips=('trips', 'sum')
										)
										total_trips_sliced_g = grouped['trips'].sum()
										
										# Unit test: check total number of trips up to the fourth decimal place
										key = f"{date.date()}_{professional}_{purpose}_{income}_{age}"
										assert total_trips_sliced.round(4) == total_trips_sliced_g.round(4), (
										f"Mismatch in total trips for date {key}: "
										f"before={round(total_trips_sliced,0)}, after={round(total_trips_sliced_g,0)}"
										)
										matrix = grouped.reset_index().pivot(index = 'macro_origin', columns = 'macro_destination', values = 'trips').fillna(0)
										# Reindex so that all zones are represented in the matrix, even if a flow is empty
										matrix = matrix.reindex(index=all_origins, columns=all_origins, fill_value=0)
										matrices[key] = matrix
										trips = matrix.values.sum()
										print(f"Completed {key}. Total trips: {round(trips,0)}")
										total_trips_seg += trips
										

			elif professional != 'No': 	# Unique values: "urban" and "interurban"
				total_trips_Non_prof = sliced_df['trips'].sum()
				
				# reshape
				grouped = sliced_df.groupby(['macro_origin', 'macro_destination']).agg(
				trips=('trips', 'sum')
				)
				total_trips_Non_prof_g = grouped['trips'].sum()
				# Unit test: check total number of trips up to the fourth decimal place
				key = f"{date.date()}_Professionals"
				assert total_trips_Non_prof.round(4) == total_trips_Non_prof_g.round(4), (
				f"Mismatch in total trips for date {key}: "
				f"before={round(total_trips_Non_prof,4)}, after={round(total_trips_Non_prof_g,0)}"
				)
				matrix = grouped.reset_index().pivot(index = 'macro_origin', columns = 'macro_destination', values = 'trips').fillna(0)
				matrix = matrix.reindex(index=all_origins, columns=all_origins, fill_value=0)
				matrices[key] = matrix
				trips = matrix.values.sum()
				print(f"Completed {key}. Total trips: {round(trips,0)}")
				total_trips_seg += trips
	
	print(total_trips_seg)
	assert round(day_total, 4) ==	round(total_trips_seg, 4), (
		f"Matrix slicing didn't work: original total: {round(day_total,0)}, sliced: {round(total_trips_seg,0)}"
	)

# display(matrices[key].head())

Total trips: 103327782.0, total trips in rows without any N/A: 102194566.0
Trips lost: 1133216.0
Completed overall matrix for date:2023-01-08 00:00:00. Total trips: 102194566.0
Completed 2023-01-08_No_O_I4_65-100. Total trips: 2076063.0
Completed 2023-01-08_No_O_I4_25-45. Total trips: 3002037.0
Completed 2023-01-08_No_O_I4_45-65. Total trips: 3406655.0
Completed 2023-01-08_No_O_I4_0-25. Total trips: 2456213.0
Completed 2023-01-08_No_O_I2_25-45. Total trips: 4743047.0
Completed 2023-01-08_No_O_I2_65-100. Total trips: 2807463.0
Completed 2023-01-08_No_O_I2_0-25. Total trips: 3911207.0
Completed 2023-01-08_No_O_I2_45-65. Total trips: 5250251.0
Completed 2023-01-08_No_O_I1_25-45. Total trips: 2817191.0
Completed 2023-01-08_No_O_I1_45-65. Total trips: 2952934.0
Completed 2023-01-08_No_O_I1_65-100. Total trips: 1512485.0
Completed 2023-01-08_No_O_I1_0-25. Total trips: 2456552.0
Completed 2023-01-08_No_O_I3_45-65. Total trips: 6157943.0
Completed 2023-01-08_No_O_I3_0-25. Total trips: 4431818.

In [90]:
# delete previous results
import shutil
shutil.rmtree(output_folder)

# create folder
if not os.path.exists(output_folder):
    os.mkdir(output_folder)

# export each element of the dictionary (date) as a separate dataframe
for key, matrix in day_matrices.items():
    filename = f"matrix_{key}.csv"
    filepath = os.path.join(output_folder,filename)

    matrix.to_csv(filepath)
    print(f"Exported {filename} to {output_folder}")
    
# export each element of the dictionary (date) as a separate dataframe
for key, matrix in matrices.items():
    filename = f"matrix_{key}.csv"
    filepath = os.path.join(output_folder,filename)

    matrix.to_csv(filepath)
    print(f"Exported {filename} to {output_folder}")

Exported matrix_2023-01-08.csv to D:/Ctym/p/p174_GemeloDigital_Leon/5_Trabajo/dev/outputs
Exported matrix_2023-01-21.csv to D:/Ctym/p/p174_GemeloDigital_Leon/5_Trabajo/dev/outputs
Exported matrix_2023-02-24.csv to D:/Ctym/p/p174_GemeloDigital_Leon/5_Trabajo/dev/outputs
Exported matrix_2023-05-24.csv to D:/Ctym/p/p174_GemeloDigital_Leon/5_Trabajo/dev/outputs
Exported matrix_2023-07-19.csv to D:/Ctym/p/p174_GemeloDigital_Leon/5_Trabajo/dev/outputs
Exported matrix_2023-08-25.csv to D:/Ctym/p/p174_GemeloDigital_Leon/5_Trabajo/dev/outputs
Exported matrix_2023-08-26.csv to D:/Ctym/p/p174_GemeloDigital_Leon/5_Trabajo/dev/outputs
Exported matrix_2023-11-17.csv to D:/Ctym/p/p174_GemeloDigital_Leon/5_Trabajo/dev/outputs
Exported matrix_2023-12-04.csv to D:/Ctym/p/p174_GemeloDigital_Leon/5_Trabajo/dev/outputs
Exported matrix_2023-01-08_No_O_I4_65-100.csv to D:/Ctym/p/p174_GemeloDigital_Leon/5_Trabajo/dev/outputs
Exported matrix_2023-01-08_No_O_I4_25-45.csv to D:/Ctym/p/p174_GemeloDigital_Leon/5_T